<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_05/src/part_2/answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    pass  # This notebook does not require additional packages.
else:
    print('Local environment: Make sure `conda activate oop_practice` is active.')

# Lecture 5 — Part 2 (Slides 15–35): ABCs, Operators, Metaclasses & a K-NN Case Study

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. Use `__subclasshook__` to make duck-typed classes pass `isinstance()`.
2. Override `__add__` / `__radd__` to overload the `+` operator.
3. Recognize *why* subclassing built-in `list` directly is a trap.
4. Extend `list` safely with `super()`.
5. Walk through Python's 6-step class creation lifecycle.
6. Define a custom metaclass and inject attributes.
7. Compare `abc.ABCMeta` (explicit) and `abc.ABC` (modern) declarations.
8. Build a small K-NN data-partitioning framework using `@overload`, properties, and modulo dealing.

## How this notebook is structured

Each section follows the same rhythm:

1. **Concept** — short explanation, taken directly from the slides.
2. **Full code reference** — the *goal* code block (your target).
3. **TODO** cell — fill the small blanks marked `# TODO`.
4. **Quick check** — assertions verify your solution.

The TODO blanks are intentionally tiny — usually one expression per line — so you can focus on understanding *each* line, not on writing big code from scratch.

## 1. The `__subclasshook__` Method (Slide 15)

> *"Treat unrelated classes as subclasses without inheritance — duck typing made visible to `isinstance()`."*

### Why it exists

Python's `isinstance(x, MediaLoader)` normally checks the **inheritance** chain. But sometimes a class has the *right shape* (it defines a `play()` method) without inheriting from `MediaLoader`. `__subclasshook__` lets the abstract base class say "yes, this class counts as one of mine."

### Three return values

| Return            | Meaning                                                     |
| ----------------- | ----------------------------------------------------------- |
| `True`            | C **counts as** a subclass.                                 |
| `False`           | C is **forcibly excluded**.                                 |
| `NotImplemented`  | Fall back to the **normal** MRO check (the usual default).  |

### Goal

Reproduce the slide-15 example: an abstract `MediaLoader` whose `__subclasshook__` accepts *any* class that defines a `play()` method, even if it never inherits from `MediaLoader`.

### Full code reference (Slide 15)

```python
import abc

class MediaLoader(abc.ABC):
    @abc.abstractmethod
    def play(self):
        pass

    @classmethod
    def __subclasshook__(cls, C):
        if cls is MediaLoader:
            # Check if 'play' is in any class dict in C's MRO
            if any("play" in B.__dict__ for B in C.__mro__):
                return True
        return NotImplemented
```

After this, *any* class that defines `play()` (even with no inheritance) will satisfy `isinstance(obj, MediaLoader)`.

In [ ]:
# Solution 1 — MediaLoader with __subclasshook__
import abc


class MediaLoader(abc.ABC):
    @abc.abstractmethod
    def play(self):
        pass

    @classmethod
    def __subclasshook__(cls, C):
        if cls is MediaLoader:
            if any("play" in B.__dict__ for B in C.__mro__):
                return True
        return NotImplemented

In [ ]:
# Quick check
class Mp3Player:                       # NOTE: NO inheritance from MediaLoader
    def play(self):
        return "mp3 playing"


class SilentBox:                       # NO play() method
    pass


mp3 = Mp3Player()
box = SilentBox()

assert isinstance(mp3, MediaLoader), "Mp3Player has play(), should pass isinstance"
assert not isinstance(box, MediaLoader), "SilentBox has no play(), should fail"
print("OK: duck-typed Mp3Player passes isinstance(_, MediaLoader)")
print("OK: SilentBox correctly rejected")
print("Section 1 passed.")

## 2. Operator Overloading — `__add__` and `__radd__` (Slides 16–17)

### 2.1  The concept (Slide 16)

Python operators are implemented by **special ("magic") methods** on classes.

| Operator | Magic method            | Reverse method            |
| -------- | ----------------------- | ------------------------- |
| `+`      | `__add__(self, o)`      | `__radd__(self, o)`       |
| `-`      | `__sub__(self, o)`      | `__rsub__(self, o)`       |
| `/`      | `__truediv__(self, o)`  | `__rtruediv__(self, o)`   |

**Fallback rule (precise):** for `a + b`, Python first calls `type(a).__add__(a, b)`.

- If it is **missing** *or* returns `NotImplemented`, Python then tries `type(b).__radd__(b, a)`.
- ⚠️ Returning `NotImplemented` is *not* the same as raising an exception.

### 2.2  Dice + Die (Slide 17)

The slide-17 example: a `Dice` collection that supports adding a single `Die` from **either** side:

```
Dice(d1, d2, d3) + d4   # uses __add__   → 4 dice
d4 + Dice(d1, d2, d3)   # uses __radd__  → 4 dice
```

Two rules from the slide:

1. **Returns a *new* `Dice`** — does *not* mutate `self` (Python `+` convention).
2. **`'Dice'` in quotes** is a *forward reference* — the class is still being defined when the annotation is read.

### Full code reference (Slide 17)

```python
class Die:
    def __init__(self, sides: int = 6) -> None:
        self.sides = sides

    def __repr__(self) -> str:
        return f"Die({self.sides})"


class Dice:
    def __init__(self, *dice: Die) -> None:
        self.dice = list(dice)

    def __add__(self, die: Die) -> 'Dice':
        # Dice() + Die()
        return Dice(*self.dice, die)

    def __radd__(self, die: Die) -> 'Dice':
        # Die() + Dice()
        return self.__add__(die)

    def __repr__(self) -> str:
        return f"Dice({len(self.dice)} dice)"
```

In [ ]:
# Solution 2 — Dice with __add__ and __radd__

class Die:
    def __init__(self, sides: int = 6) -> None:
        self.sides = sides

    def __repr__(self) -> str:
        return f"Die({self.sides})"


class Dice:
    def __init__(self, *dice: Die) -> None:
        self.dice = list(dice)

    def __add__(self, die: Die) -> 'Dice':
        return Dice(*self.dice, die)

    def __radd__(self, die: Die) -> 'Dice':
        return self.__add__(die)

    def __repr__(self) -> str:
        return f"Dice({len(self.dice)} dice)"

In [ ]:
# Quick check
d1, d2, d3, d4 = Die(), Die(), Die(), Die()
my_dice = Dice(d1, d2, d3)

new_a = my_dice + d4    # __add__
new_b = d4 + my_dice    # __radd__ (Die has no __add__ for Dice → falls back)

assert len(my_dice.dice) == 3, "self should NOT be mutated"
assert len(new_a.dice) == 4
assert len(new_b.dice) == 4
print(f"OK: my_dice still has {len(my_dice.dice)} dice (not mutated)")
print(f"OK: my_dice + d4 → {new_a}")
print(f"OK: d4 + my_dice → {new_b}")
print("Section 2 passed.")

## 3. Subclassing Built-in `list` — The Trap (Slides 18–19)

**The pitfall:** if you override `append()` on a class that inherits from `list`, **`extend()`, `__init__`, and `+=` silently bypass your override.**

Why? `list` is implemented in **C**. Its other methods call the C-level append directly — *not* the Python-level method you overrode.

### Demonstration (Slide 19)

```python
class LoudList(list):
    def append(self, item):
        print(f"appending {item}")
        super().append(item)

L = LoudList([1, 2, 3])   # silent — append never called
L.extend([4, 5])          # silent — extend bypasses append
L += [6, 7]               # silent — __iadd__ also bypasses
L.append(8)               # prints "appending 8"  ✓
```

You will run exactly that code below and **observe** the silent paths.

### Full code reference (Slide 19)

The code below is purely an *observation cell* — there is no TODO, just run it and read the output.

```python
class LoudList(list):
    def append(self, item):
        print(f"appending {item}")
        super().append(item)
```

In [ ]:
# Observation: subclassing list directly is leaky.
class LoudList(list):
    def append(self, item):
        print(f"appending {item}")
        super().append(item)


print("→ LoudList([1, 2, 3])")
L = LoudList([1, 2, 3])
print(f"   contents: {list(L)}\n")

print("→ L.extend([4, 5])")
L.extend([4, 5])
print(f"   contents: {list(L)}\n")

print("→ L += [6, 7]")
L += [6, 7]
print(f"   contents: {list(L)}\n")

print("→ L.append(8)   # the only path that prints")
L.append(8)
print(f"   contents: {list(L)}")

**Take-away (slide 19, bottom):**

> *list is implemented in C. Its other methods call the C-level append directly — not the Python-level method you overrode.*
>
> **Fix:** inherit from `collections.UserList` — pure Python, all paths go through your override.

## 4. Extending `list` Safely with `super()` (Slide 20)

If you *must* inherit from built-in `list`, the slide-20 pattern is to be **explicit** about delegation: every override calls `super().<method>()` so the C-level path still runs, while you add your own behavior on top.

### Goal

Build `CustomSequence(list)` that:

- Accepts an optional iterable in `__init__` (defaults to empty).
- Prints `"Appending <item>!"` whenever `append` is called, then delegates to `super().append`.

### Full code reference (Slide 20)

```python
from typing import Iterable, Optional, Any

class CustomSequence(list):
    def __init__(
        self,
        iterable: Optional[Iterable[Any]] = None
    ) -> None:
        if iterable:
            super().__init__(iterable)
        else:
            super().__init__()

    def append(self, item: Any) -> None:
        print(f"Appending {item}!")
        super().append(item)
```

In [ ]:
# Solution 4 — CustomSequence
from typing import Iterable, Optional, Any


class CustomSequence(list):
    def __init__(
        self,
        iterable: Optional[Iterable[Any]] = None
    ) -> None:
        if iterable:
            super().__init__(iterable)
        else:
            super().__init__()

    def append(self, item: Any) -> None:
        print(f"Appending {item}!")
        super().append(item)

In [ ]:
# Quick check
seq = CustomSequence([10, 20])
assert list(seq) == [10, 20]
print(f"OK: initial contents {list(seq)}")

print("→ seq.append(30)   # should print 'Appending 30!'")
seq.append(30)
assert list(seq) == [10, 20, 30]

empty = CustomSequence()
assert list(empty) == []
print("OK: empty initialisation works")
print("Section 4 passed.")

## 5. The Class Creation Lifecycle & Defining a Metaclass (Slides 21–22)

A *metaclass* is a class **whose instances are classes**. The default metaclass is `type`.

### The 6-step lifecycle (slide 21)

| Step | What happens                                                        |
| ---- | ------------------------------------------------------------------- |
| 1    | **Parse header** — `class C(metaclass=M):`                          |
| 2    | `M.__prepare__()` — returns the namespace dict                      |
| 3    | **Body execution** — class body runs *inside* that namespace        |
| 4    | `M.__new__()` — builds the class object                             |
| 5    | `M.__init__()` — initializes the class                              |
| 6    | `M.__call__()` — runs on **each instantiation** of the class         |

> Note: `__prepare__` runs **before** body execution. All four can be overridden.

### Goal (slide 22)

Define `SpecialMeta(type)` that injects an attribute `injected_by_meta = True` into every class that uses it as its metaclass.

### Full code reference (Slide 22)

```python
class SpecialMeta(type):
    def __new__(mcs, name, bases, namespace, **kwargs):
        print(f"Creating class {name}...")
        # Inject an attribute automatically
        namespace['injected_by_meta'] = True
        return super().__new__(mcs, name, bases, namespace)


class MyClass(metaclass=SpecialMeta):
    pass

print(MyClass.injected_by_meta)   # Output: True
```

In [ ]:
# Solution 5 — SpecialMeta

class SpecialMeta(type):
    def __new__(mcs, name, bases, namespace, **kwargs):
        print(f"Creating class {name}...")
        namespace['injected_by_meta'] = True
        return super().__new__(mcs, name, bases, namespace)

In [ ]:
# Quick check
class MyClass(metaclass=SpecialMeta):
    pass

class AnotherClass(metaclass=SpecialMeta):
    pass

assert MyClass.injected_by_meta is True
assert AnotherClass.injected_by_meta is True
print(f"OK: MyClass.injected_by_meta = {MyClass.injected_by_meta}")
print("Section 5 passed.")

## 6. Abstract Metaclasses — `abc.ABCMeta` vs `abc.ABC` (Slide 23)

`abc.ABCMeta` is a built-in metaclass that **enforces abstract methods**. Inheriting from `abc.ABC` is just a cleaner way to get the same metaclass.

### Connecting to the lifecycle

- `ABCMeta.__new__` (step 4) scans the namespace and collects every `@abstractmethod` into `__abstractmethods__`.
- `__call__` (step 6) blocks instantiation if any abstract method remains unimplemented.

### Goal (slide 23)

Declare an abstract `Die` class with an abstract `roll()` method **two equivalent ways**, and verify both behave identically.

### Full code reference (Slide 23)

```python
import abc

# Approach 1 — Explicit (under the hood)
class ExplicitDie(metaclass=abc.ABCMeta):
    @abc.abstractmethod
    def roll(self):
        pass


# Approach 2 — Modern (preferred)
class ModernDie(abc.ABC):
    @abc.abstractmethod
    def roll(self):
        pass
```

Both produce the **same** class object — both have `__abstractmethods__ == {'roll'}`, and instantiating either directly raises `TypeError`.

In [ ]:
# Solution 6 — Two equivalent ways to declare an abstract Die
import abc


# Approach 1 — Explicit
class ExplicitDie(metaclass=abc.ABCMeta):
    @abc.abstractmethod
    def roll(self):
        pass


# Approach 2 — Modern
class ModernDie(abc.ABC):
    @abc.abstractmethod
    def roll(self):
        pass

In [ ]:
# Quick check
assert ExplicitDie.__abstractmethods__ == frozenset({"roll"})
assert ModernDie.__abstractmethods__ == frozenset({"roll"})
print(f"OK: ExplicitDie.__abstractmethods__ = {set(ExplicitDie.__abstractmethods__)}")
print(f"OK: ModernDie.__abstractmethods__   = {set(ModernDie.__abstractmethods__)}")

for cls in (ExplicitDie, ModernDie):
    try:
        cls()
    except TypeError as e:
        print(f"OK: cannot instantiate {cls.__name__} directly — {e}")
    else:
        raise AssertionError(f"{cls.__name__} should not be instantiable")

print("Section 6 passed.")

## 7. Case Study — K-NN Train/Test Partitioning (Slides 24–34)

### The problem (slide 24)

We want to split raw sample data into a **Training set** and a **Testing set**. The split logic should be **encapsulated** in a class that *behaves like a list*.

### Two strategies

| Strategy | Idea                                                          |
| -------- | ------------------------------------------------------------- |
| 1        | **Shuffle** the data, then **slice** at a position.           |
| 2        | **Deal** items incrementally — 8 to training, 2 to testing.   |

### Class hierarchy (slide 25)

```
                       list[dict]   (built-in)
                            ▲
                            │
                  «abstract» SamplePartition
                            ▲
              ┌─────────────┴─────────────┐
              │                           │
   ShufflingSamplePartition       «abstract» DealingPartition
       (Strategy 1)                          ▲
                                             │
                                  CountingDealingPartition
                                       (Strategy 2)
```

We will build this in 4 small steps:

- **7.1** abstract `SamplePartition` with `@overload`-typed `__init__`.
- **7.2** concrete `ShufflingSamplePartition` (Strategy 1).
- **7.3** anti-pattern warning (read-only).
- **7.4** abstract `DealingPartition` + concrete `CountingDealingPartition` (Strategy 2).

### 7.1 — `SamplePartition` with `@overload` (Slides 26–28)

Python supports **only one** real implementation of `__init__`. But `list` accepts multiple call patterns:

- `list()` — empty
- `list(iterable)` — from an iterable

The `@overload` decorator (from `typing`) declares **multiple typed signatures** so type checkers know which calls are valid. The stubs are erased at runtime — only the last `def` is callable. **You write one real dispatch yourself.**

### Full code reference (Slide 28)

```python
from typing import List, Iterable, Optional, overload
import abc


class SamplePartition(List[dict], abc.ABC):
    @overload
    def __init__(self, *, training_subset: float = 0.80) -> None: ...

    @overload
    def __init__(
        self,
        iterable: Optional[Iterable[dict]] = None,
        *,
        training_subset: float = 0.80,
    ) -> None: ...

    def __init__(self, iterable=None, *, training_subset=0.80):
        self.training_subset = training_subset
        if iterable:
            super().__init__(iterable)
        else:
            super().__init__()
```

In [ ]:
# Solution 7.1 — SamplePartition with @overload
from typing import List, Iterable, Optional, overload
import abc


class SamplePartition(List[dict], abc.ABC):
    @overload
    def __init__(self, *, training_subset: float = 0.80) -> None: ...

    @overload
    def __init__(
        self,
        iterable: Optional[Iterable[dict]] = None,
        *,
        training_subset: float = 0.80,
    ) -> None: ...

    def __init__(self, iterable=None, *, training_subset=0.80):
        self.training_subset = training_subset
        if iterable:
            super().__init__(iterable)
        else:
            super().__init__()

In [ ]:
# Quick check
# Slide 28 declares no @abstractmethod on SamplePartition itself, so it is technically
# instantiable. The «abstract» label in the slide-25 diagram is a *design intent*,
# made concrete by the abstract methods that subclasses (DealingPartition, etc.) add
# in section 7.4. Here we just verify the @overload-typed __init__ dispatch works.

class _Probe(SamplePartition):
    pass


p_empty = _Probe()
assert list(p_empty) == []
assert p_empty.training_subset == 0.80

p_full = _Probe([{"x": 1}, {"x": 2}], training_subset=0.5)
assert len(p_full) == 2
assert p_full.training_subset == 0.5
print("OK: SamplePartition initialises both empty and from iterable.")
print("Section 7.1 passed.")

### 7.2 — `ShufflingSamplePartition` (Slides 29–30)

Strategy 1 — **Shuffle and slice.**

- `split` is the integer index where training ends and testing begins, derived from `training_subset`.
- `training` shuffles the whole list and returns the prefix `[:split]`.
- `testing` shuffles and returns the suffix `[split:]`.

> ⚠️ Slide 32 will explain why this design is actually an *anti-pattern* — but we still implement it as the lecture does, then read the warning.

### Visual (slide 29)

```
Before:         A B C D E F G H I J
After shuffle:  D H A J F B I G C E
Sliced @ 7:    [D H A J F B I] [G C E]
                 training         testing
```

### Full code reference (Slide 30)

```python
import random


class ShufflingSamplePartition(SamplePartition):
    @property
    def split(self) -> int:
        return int(len(self) * self.training_subset)

    @property
    def training(self) -> list:
        random.shuffle(self)
        return self[: self.split]

    @property
    def testing(self) -> list:
        random.shuffle(self)
        return self[self.split :]
```

In [ ]:
# Solution 7.2 — ShufflingSamplePartition
import random


class ShufflingSamplePartition(SamplePartition):
    @property
    def split(self) -> int:
        return int(len(self) * self.training_subset)

    @property
    def training(self) -> list:
        random.shuffle(self)
        return self[: self.split]

    @property
    def testing(self) -> list:
        random.shuffle(self)
        return self[self.split :]

In [ ]:
# Quick check
random.seed(42)  # reproducible for the assertion below

raw = [{"i": i} for i in range(10)]
ssp = ShufflingSamplePartition(raw, training_subset=0.7)

assert ssp.split == 7, f"expected split = 7, got {ssp.split}"
train = ssp.training
test = ssp.testing
assert len(train) == 7
assert len(test) == 3
print(f"OK: split = {ssp.split}, training has {len(train)}, testing has {len(test)}")
print("Section 7.2 passed.")

### 7.3 — Why the Shuffling Strategy Is an Anti-Pattern (Slide 32)

> ⚠️ **The `ShufflingSamplePartition` shuffles the list every time you read `.training` or `.testing`.**

**Consequences:**

- Calling `p.training` then `p.testing` reshuffles in between → **train/test leakage** (the same item can land in both).
- Each access produces a different result → **no reproducibility**.
- Properties are expected to be **pure** — mutating `self` inside a getter violates that contract.

**Fix:** shuffle **once** in `__init__`, cache the split indices, and seed with `random.Random(seed)` for reproducibility.

The next strategy — *Dealing* — avoids this entirely.

### 7.4 — Incremental Dealing Strategy (Slides 31, 33–34)

Like a card dealer:

- For every 10 items dealt, **8 → training**, **2 → testing**.
- Decision uses **modulo math**: `if counter % 10 < 8: training, else testing`.
- Single pass, no shuffle, **deterministic** — no leakage, fully reproducible.

```
counter %10:  0 1 2 3 4 5 6 7 | 8 9
              ┌─train───────┐ ┌test─┐
```

We will:

1. Define a tiny abstract `DealingPartition(SamplePartition)` — slide 25 marks it «abstract» with `append() → deal`, so we declare `append` as the abstract method.
2. Define the concrete `CountingDealingPartition` exactly as slide 33 shows.

### Full code reference (Slide 33)

```python
class DealingPartition(SamplePartition):
    @abc.abstractmethod
    def append(self, item) -> None: ...


class CountingDealingPartition(DealingPartition):
    def __init__(self, items=None, *, training_subset=(8, 10)):
        self.training_subset = training_subset
        self.counter = 0
        self._training: list = []
        self._testing: list = []
        if items:
            self.extend(items)

    def append(self, item) -> None:
        n, d = self.training_subset
        if self.counter % d < n:
            self._training.append(item)
        else:
            self._testing.append(item)
        self.counter += 1
```

> Note: the slide writes `self.extend(items)` in `__init__`, but as section 3 demonstrated, `list.extend` *bypasses* our Python-level `append` override. In practice you would feed items one at a time via `append()` — exactly how the dealing strategy is described on slide 31 ("processes data iteratively").

In [ ]:
# Solution 7.4 — DealingPartition (abstract) and CountingDealingPartition (concrete)

class DealingPartition(SamplePartition):
    @abc.abstractmethod
    def append(self, item) -> None: ...


class CountingDealingPartition(DealingPartition):
    def __init__(self, items=None, *, training_subset=(8, 10)):
        self.training_subset = training_subset
        self.counter = 0
        self._training: list = []
        self._testing: list = []
        if items:
            self.extend(items)

    def append(self, item) -> None:
        n, d = self.training_subset
        if self.counter % d < n:
            self._training.append(item)
        else:
            self._testing.append(item)
        self.counter += 1

In [ ]:
# Quick check — the slide-34 example: items = [A, B, ..., J] → train [A..H], test [I, J]
# NOTE: the dealing strategy is meant to *receive* items one at a time (Slide 31:
# "processes data iteratively"). We therefore feed them by calling append() in a loop —
# this is exactly the C-level extend pitfall from Section 3 in action: passing items via
# `extend` would silently bypass our Python-level append() override.

items = [{"id": c} for c in "ABCDEFGHIJ"]
cdp = CountingDealingPartition()
for it in items:
    cdp.append(it)

train_ids = [d["id"] for d in cdp._training]
test_ids = [d["id"] for d in cdp._testing]

assert train_ids == list("ABCDEFGH"), f"got {train_ids}"
assert test_ids == list("IJ"), f"got {test_ids}"
print(f"OK: training = {train_ids}")
print(f"OK: testing  = {test_ids}")

# DealingPartition declares 'append' as abstract (the structural contract from Slide 25).
# Note: Python's C-level list.__new__ does NOT enforce __abstractmethods__ at instantiation
# time — that runtime block only fires for object.__new__-based hierarchies. So the contract
# here lives in __abstractmethods__ rather than as a TypeError on construction.
assert "append" in DealingPartition.__abstractmethods__
print(f"OK: DealingPartition.__abstractmethods__ = {set(DealingPartition.__abstractmethods__)}")

print("Section 7.4 passed.")

## 8. Summary (Slide 35)

> **Abstract Base Classes establish robust structural contracts.**
>
> By leveraging the `abc` and `collections.abc` modules, operator overloading, and metaclasses, we transition from writing isolated scripts to designing cohesive, highly integrated OOP frameworks that feel native to Python.

### What you practiced

| Slide | Topic                                              | What you built                                  |
| ----- | -------------------------------------------------- | ------------------------------------------------ |
| 15    | `__subclasshook__`                                 | `MediaLoader` (duck-typed `isinstance`)          |
| 16–17 | Operator overloading + `__add__` / `__radd__`      | `Dice + Die`, `Die + Dice`                       |
| 18–20 | Subclassing `list` — pitfall + safe pattern        | `LoudList` (observation), `CustomSequence`       |
| 21–22 | Class creation lifecycle, custom metaclass         | `SpecialMeta`                                    |
| 23    | Abstract metaclasses                               | `ExplicitDie`, `ModernDie`                       |
| 24–34 | K-NN train/test split case study                   | `SamplePartition`, `ShufflingSamplePartition`, `CountingDealingPartition` |

Great work! 🎯